# Autoencoders

**Learning Objectives:** Understand autoencoder architecture (encoder-decoder), undercomplete vs overcomplete, denoising autoencoders, VAEs conceptually, image reconstruction, anomaly detection, and latent space visualization.

## 1. Autoencoder Architecture

An autoencoder learns to compress (encode) data into a lower-dimensional latent space and reconstruct (decode) it back.

**Components:**
- **Encoder:** $z = f_\phi(x)$ — compresses input to latent code
- **Decoder:** $\hat{x} = g_\theta(z)$ — reconstructs input from latent code
- **Loss:** $\mathcal{L} = ||x - \hat{x}||^2$ — reconstruction error

The latent space $z$ is a **bottleneck** that forces the model to learn efficient representations.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Build a simple autoencoder for MNIST
input_dim = 784
encoding_dim = 32  # bottleneck size

# Encoder
input_img = keras.Input(shape=(input_dim,))
encoded = layers.Dense(128, activation='relu')(input_img)
encoded = layers.Dense(64, activation='relu')(encoded)
encoded = layers.Dense(encoding_dim, activation='relu')(encoded)

# Decoder
decoded = layers.Dense(64, activation='relu')(encoded)
decoded = layers.Dense(128, activation='relu')(decoded)
decoded = layers.Dense(input_dim, activation='sigmoid')(decoded)

autoencoder = keras.Model(input_img, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

# Separate encoder for later use
encoder = keras.Model(input_img, encoded)

autoencoder.summary()

In [ ]:
# Load and prepare MNIST
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

print(f"Train: {x_train.shape}, Test: {x_test.shape}")

In [ ]:
# Train the autoencoder
history = autoencoder.fit(
    x_train, x_train,  # input = target (reconstruction)
    batch_size=256,
    epochs=20,
    validation_data=(x_test, x_test),
    verbose=1
)

plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Test')
plt.title('Autoencoder Reconstruction Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 2. Undercomplete vs Overcomplete Autoencoders

| Type | Bottleneck Dimension | Behavior |
|------|---------------------|----------|
| **Undercomplete** | $\ll$ input dim | Learns compressed representation (useful) |
| **Overcomplete** | $\geq$ input dim | May learn identity (needs regularization) |

An undercomplete autoencoder (encoding dim < input dim) forces the model to learn meaningful features. Overcomplete autoencoders require regularization (sparse, denoising) to be useful.

In [ ]:
# Visualize reconstructions
decoded_imgs = autoencoder.predict(x_test[:10])

fig, axes = plt.subplots(2, 10, figsize=(20, 4))
for i in range(10):
    # Original
    axes[0, i].imshow(x_test[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original')
    # Reconstructed
    axes[1, i].imshow(decoded_imgs[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Reconstructed')
plt.tight_layout()
plt.show()

## 3. Denoising Autoencoder

A denoising autoencoder learns to remove noise by reconstructing clean images from corrupted inputs.

In [ ]:
# Build denoising autoencoder
def add_noise(images, noise_factor=0.3):
    noisy = images + noise_factor * np.random.randn(*images.shape)
    return np.clip(noisy, 0.0, 1.0)

x_train_noisy = add_noise(x_train)
x_test_noisy = add_noise(x_test)

# Same architecture but trained with noisy input -> clean output
denoising_ae = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(784, activation='sigmoid')
])
denoising_ae.compile(optimizer='adam', loss='mse')

denoising_ae.fit(x_train_noisy, x_train,  # noisy -> clean
                 batch_size=256, epochs=15,
                 validation_data=(x_test_noisy, x_test),
                 verbose=1)

In [ ]:
# Visualize denoising results
denoised = denoising_ae.predict(x_test_noisy[:10])

fig, axes = plt.subplots(3, 10, figsize=(20, 6))
titles = ['Noisy Input', 'Denoised Output', 'Original']
for i in range(10):
    axes[0, i].imshow(x_test_noisy[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(denoised[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    axes[2, i].imshow(x_test[i].reshape(28, 28), cmap='gray')
    axes[2, i].axis('off')
axes[0, 0].set_title('Noisy')
axes[1, 0].set_title('Denoised')
axes[2, 0].set_title('Original')
plt.tight_layout()
plt.show()

## 4. Autoencoders for Anomaly Detection

Autoencoders detect anomalies via reconstruction error:
- **Normal samples:** Low reconstruction error (model has seen similar patterns)
- **Anomalies:** High reconstruction error (model can't reconstruct unusual patterns)

In [ ]:
# Anomaly detection on MNIST with AE
# Treat digit '0' as normal, other digits as anomalies
x_train_0 = x_train[y_train == 0]
x_test_0 = x_test[y_test == 0]
x_test_others = x_test[y_test != 0]

print(f"Train (digit 0): {x_train_0.shape}")
print(f"Test (digit 0): {x_test_0.shape}")
print(f"Test (others): {x_test_others.shape}")

# Train AE on digit '0' only
anomaly_ae = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(784,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(784, activation='sigmoid')
])
anomaly_ae.compile(optimizer='adam', loss='mse')
anomaly_ae.fit(x_train_0, x_train_0, batch_size=128, epochs=10, verbose=0)

# Compute reconstruction errors
recon_0 = np.mean((x_test_0 - anomaly_ae.predict(x_test_0)) ** 2, axis=1)
recon_others = np.mean((x_test_others - anomaly_ae.predict(x_test_others)) ** 2, axis=1)

print(f"Mean reconstruction error (digit 0): {np.mean(recon_0):.4f}")
print(f"Mean reconstruction error (others): {np.mean(recon_others):.4f}")

plt.figure(figsize=(10, 4))
plt.hist(recon_0, bins=30, alpha=0.6, label='Digit 0 (Normal)')
plt.hist(recon_others, bins=30, alpha=0.6, label='Other Digits (Anomaly)')
plt.xlabel('Reconstruction Error')
plt.ylabel('Count')
plt.title('Anomaly Detection with Autoencoder')
plt.legend()
plt.show()

## 5. Variational Autoencoders (VAEs) — Conceptual

VAEs are generative models that learn the distribution of the latent space, not just a deterministic encoding.

**Key difference from standard AE:**
- Standard AE: encoder outputs $z$ directly (deterministic)
- VAE: encoder outputs $\mu$ and $\sigma$ (parameters of a Gaussian), then samples $z \sim \mathcal{N}(\mu, \sigma^2)$

**VAE Loss:** $\mathcal{L} = \underbrace{||x - \hat{x}||^2}_{\text{reconstruction}} + \beta \underbrace{D_{KL}(\mathcal{N}(\mu, \sigma^2) || \mathcal{N}(0, 1))}_{\text{KL divergence}}$

In [ ]:
# VAE implementation in Keras
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# Encoder
latent_dim = 2
encoder_inputs = keras.Input(shape=(784,))
x = layers.Dense(128, activation='relu')(encoder_inputs)
x = layers.Dense(64, activation='relu')(x)
z_mean = layers.Dense(latent_dim, name='z_mean')(x)
z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
z = Sampling()([z_mean, z_log_var])

vae_encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z])

# Decoder
decoder_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(64, activation='relu')(decoder_inputs)
x = layers.Dense(128, activation='relu')(x)
decoder_outputs = layers.Dense(784, activation='sigmoid')(x)

vae_decoder = keras.Model(decoder_inputs, decoder_outputs)

# VAE (end-to-end)
outputs = vae_decoder(z)
vae = keras.Model(encoder_inputs, outputs)

# Add KL loss
kl_loss = -0.5 * tf.reduce_mean(z_log_var - tf.square(z_mean) - tf.exp(z_log_var) + 1)
vae.add_loss(kl_loss)
vae.compile(optimizer='adam', loss='mse')

print("VAE model built (latent_dim=2).")

In [ ]:
# Train VAE (for demo, train briefly)
vae.fit(x_train, x_train, batch_size=256, epochs=10, validation_data=(x_test, x_test), verbose=1)

In [ ]:
# Latent space visualization
z_means, _, _ = vae_encoder.predict(x_test, verbose=0)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(z_means[:, 0], z_means[:, 1], c=y_test, cmap='tab10', alpha=0.6, s=5)
plt.colorbar(scatter, label='Digit')
plt.title('VAE Latent Space (2D)')
plt.xlabel('z[0]')
plt.ylabel('z[1]')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Generate new digits by sampling latent space
n = 15
digit_size = 28
figure = np.zeros((digit_size * n, digit_size * n))

grid_x = np.linspace(-3, 3, n)
grid_y = np.linspace(-3, 3, n)

for i, yi in enumerate(grid_x):
    for j, xi in enumerate(grid_y):
        z_sample = np.array([[xi, yi]])
        x_decoded = vae_decoder.predict(z_sample, verbose=0)
        digit = x_decoded[0].reshape(digit_size, digit_size)
        figure[i * digit_size: (i + 1) * digit_size,
               j * digit_size: (j + 1) * digit_size] = digit

plt.figure(figsize=(10, 10))
plt.imshow(figure, cmap='gray')
plt.title('Generated Digits from VAE Latent Space')
plt.axis('off')
plt.show()

## 6. Interview Problems

### Problem 1: Image Denoising with Autoencoder

Train a convolutional autoencoder for image denoising on Fashion-MNIST. Add Gaussian noise and show that the denoising AE can recover clean images.

### Problem 2: Anomaly Detection on MNIST

Train an autoencoder on only digit '0'. Use reconstruction error to detect anomalous digits (non-zero). Report precision and recall.

### Problem 3: Latent Space Interpolation

For a trained autoencoder, interpolate between two latent vectors of different digits and visualize the smooth morphing between them.

In [ ]:
# Solution: Problem 3 — Latent space interpolation
def interpolate_latent(ae_encoder, ae_decoder, img1, img2, steps=10):
    z1 = ae_encoder.predict(img1.reshape(1, -1), verbose=0)[0]
    z2 = ae_encoder.predict(img2.reshape(1, -1), verbose=0)[0]

    alphas = np.linspace(0, 1, steps)
    interpolated = []
    for alpha in alphas:
        z = (1 - alpha) * z1 + alpha * z2
        decoded = ae_decoder.predict(z.reshape(1, -1), verbose=0)
        interpolated.append(decoded.reshape(28, 28))
    return interpolated

# Interpolate between a 3 and a 5
idx_3 = np.where(y_test == 3)[0][0]
idx_5 = np.where(y_test == 5)[0][0]

interp = interpolate_latent(encoder, autoencoder, x_test[idx_3], x_test[idx_5], steps=8)

fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for i, img in enumerate(interp):
    axes[i].imshow(img, cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f'{i}', fontsize=8)
plt.suptitle('Latent Space Interpolation: 3 → 5')
plt.show()

## Summary

- **Autoencoders** learn compressed representations via encoder-decoder architecture
- **Undercomplete** AEs (bottleneck < input dim) learn useful features
- **Denoising AEs** reconstruct clean images from corrupted inputs
- **Anomaly detection** uses reconstruction error to identify outliers
- **VAEs** learn a probabilistic latent space (Gaussian), enabling generation
- **Latent space** of well-trained AEs is continuous and semantically meaningful
- Applications: dimensionality reduction, denoising, anomaly detection, generation